# AWS Bedrock Fine-Tuning Walkthrough
Author: A Taylor

## Prerequisites

Before running this notebook, ensure you have:

1. **IAM Role** — An IAM role with Bedrock and S3 permissions (`BEDROCK_ROLE_ARN`)
2. **S3 Bucket** — A bucket for training data and model output (`S3_BUCKET`)
3. **Dataset** — The HuggingFace dataset: `Taylor658/deep-space-optical-chip-thermal-dataset`
4. **Environment** — A `.env` file with AWS credentials (see `.env.example`)

In [ ]:
# Imports and environment setup
import os
import sys
import json

from dotenv import load_dotenv

# Add project root to path
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

load_dotenv(os.path.join("..", ".env"))

from src.data_prep import DataPrepPipeline
from src.bedrock_finetune import BedrockFineTuneManager
from src.inference import BedrockInferenceClient, compare_models

print("Imports loaded — Author: A Taylor")

## Step 1 — Prepare and Upload Dataset

In [ ]:
# Prepare the dataset for Bedrock fine-tuning
pipeline = DataPrepPipeline()
pipeline.run(output_dir="../data", upload=True)

# Show sample JSONL output
sample_path = os.path.join("..", "data", "train.jsonl")
if os.path.exists(sample_path):
    with open(sample_path, "r") as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            print(json.dumps(json.loads(line), indent=2))
            print("---")

## Step 2 — Configure and Launch Bedrock Job

In [ ]:
# Launch the fine-tuning job
manager = BedrockFineTuneManager(config_path="../config/bedrock_config.yaml")
job_arn = manager.start_job()
print(f"Job ARN: {job_arn}")

## Step 3 — Monitor Job Progress

In [ ]:
# Wait for the job to complete (this will block and poll)
# For long jobs, you may prefer to check status manually:
#   status = manager.get_job_status(job_arn)

final_status = manager.wait_for_completion(job_arn, poll_interval=60)
print(f"Final status: {final_status}")

if final_status == "Completed":
    custom_model_arn = manager.get_provisioned_model_arn(job_arn)
    print(f"Custom model ARN: {custom_model_arn}")

## Step 4 — Run Inference on Fine-Tuned Model

In [ ]:
# Run inference with the fine-tuned model
# Replace with your actual custom model ARN from Step 3
finetuned_model_id = custom_model_arn  # or paste ARN string directly

client = BedrockInferenceClient(model_id=finetuned_model_id)

prompt = client.build_thermal_prompt(
    instrument="Spectrometer",
    material="Indium Phosphide",
    environment="Jovian System",
    thermal_effect="Spectral Drift",
)

print("Prompt:")
print(prompt)
print("\nResponse:")
response = client.invoke(prompt)
print(response)

## Step 5 — Compare Base vs Fine-Tuned

In [ ]:
# Compare base model vs fine-tuned model on example prompts
base_model_id = "amazon.titan-text-express-v1"

test_scenarios = [
    {
        "instrument": "Spectrometer",
        "material": "Silicon",
        "environment": "Near Earth Deep Space",
        "thermal_effect": "Spectral Drift",
    },
    {
        "instrument": "Laser Communication Terminal",
        "material": "Indium Phosphide",
        "environment": "Outer Solar System",
        "thermal_effect": "Waveguide Misalignment",
    },
    {
        "instrument": "Waveguide Sensor Array",
        "material": "Polymer",
        "environment": "Mars Transit",
        "thermal_effect": "Mechanical Cracking",
    },
]

for scenario in test_scenarios:
    prompt = BedrockInferenceClient.build_thermal_prompt(**scenario)
    result = compare_models(base_model_id, finetuned_model_id, prompt)

    print(f"\n{'='*60}")
    print(f"Scenario: {scenario['instrument']} / {scenario['material']} / {scenario['environment']}")
    print(f"{'='*60}")
    print(f"\nBase Model Response:\n{result['base_response']}")
    print(f"\nFine-Tuned Response:\n{result['finetuned_response']}")